# Importar ficheiros CRONO (cargas) para SQLite

Notebook unico com todo o processo:

1. Preparar a base de dados (tabelas `cargas_2025` / `cargas_2026` + indice unico anti-duplicados)
2. Ler os ficheiros CRONO da pasta `PASTA_FICHEIROS`
3. Inserir os dados, sem duplicar linhas
4. Validar - comparar ficheiros vs base de dados, por mes

Basta correr todas as celulas por ordem (Kernel -> Run All). Podes correr
este notebook sempre que houver ficheiros novos na pasta - os dados ja
importados nao sao duplicados, porque cada linha e considerada unica pelo
conjunto de **todas** as suas colunas.

## Configuracao

In [7]:
import os
import csv
import glob
import platform
import sqlite3
from datetime import datetime

import pandas as pd

if platform.system() == "Windows":
    DB_PATH = r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db"
    PASTA_FICHEIROS = r"C:\Users\LISARR\Documents\python\02.Pontualidade\crono"
elif platform.system() == "Darwin":
    DB_PATH = "/Volumes/RR/DB/inform_27.db"
    PASTA_FICHEIROS = "/Volumes/RR/DB/crono"
else:
    DB_PATH = "inform_27.db"
    PASTA_FICHEIROS = "crono"

print("DB_PATH:", DB_PATH)
print("PASTA_FICHEIROS:", PASTA_FICHEIROS)

DB_PATH: C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db
PASTA_FICHEIROS: C:\Users\LISARR\Documents\python\02.Pontualidade\crono


In [8]:
# Colunas do ficheiro CRONO: nome_sql -> nome original no ficheiro. O
# ficheiro tem 2 colunas chamadas "Temperatura" - por isso a leitura usa
# a posicao das colunas (nao um dict nome->valor) e a 2a fica
# TEMPERATURA_MEDIDA1, distinta de TEMPERATURA_REQUERIDA.
COLUNAS_CARGAS = {
    "TIPO_SUMINISTRO": "Tipo Suministro",
    "BASE": "Base",
    "LANZADERA": "Lanzadera",
    "PUNTO_SUMINISTRO": "Punto Suministro",
    "RUTA": "Ruta",
    "PALES_TMS": "Pal\u00e9s TMS",
    "TEMPERATURA_REQUERIDA": "Temperatura",
    "AGENCIA": "Agencia",
    "TRANSPORTISTA": "Transportista",
    "DNI": "DNI",
    "PRECINTO": "Precinto",
    "TELEFONO": "Tel\u00e9fono",
    "TRACTORA": "Tractora",
    "REMOLQUE": "Remolque",
    "FECHA_PREVISTA_POSICIONAMIENTO": "Fecha Prevista Posicionamiento",
    "HORA_PREVISTA_POSICIONAMIENTO": "Hora Prevista Posicionamiento",
    "FECHA_REAL_POSICIONAMIENTO": "Fecha Real Posicionamiento",
    "HORA_REAL_POSICIONAMIENTO": "Hora Real Posicionamiento",
    "FECHA_REAL_ENTRADA": "Fecha Real Entrada",
    "HORA_REAL_ENTRADA": "Hora Real Entrada",
    "MUELLE": "Muelle",
    "FECHA_PREVISTA_SALIDA": "Fecha Prevista Salida",
    "HORA_PREVISTA_SALIDA": "Hora Prevista Salida",
    "FECHA_REAL_SALIDA": "Fecha Real Salida",
    "HORA_REAL_SALIDA": "Hora Real Salida",
    "FECHA_PREVISTA_ENTREGA": "Fecha Prevista Entrega",
    "HORA_PREVISTA_ENTREGA": "Hora Prevista Entrega",
    "FECHA_REAL_ENTREGA": "Fecha Real Entrega",
    "HORA_REAL_ENTREGA": "Hora Real Entrega",
    "HORA_SALIDA_ENTREGA": "Hora Salida Entrega",
    "OBSERVACIONES": "Observaciones",
    "COMENTARIOS": "Comentarios",
    "ZONA": "Zona",
    "CLIENTE": "Cliente",
    "AUTORIZADO_AUTOCARGA": "Autorizado autocarga",
    "AUTOCARGA": "Autocarga",
    "HUECOS_TMS": "Huecos TMS",
    "HUECOS_CARGA": "Huecos carga",
    "HUECOS_DESCARGA": "Huecos descarga",
    "TEMPERATURA_MEDIDA1": "Temperatura",
    "TEMPERATURA_MEDIDA2": "Temperatura2",
    "MOTIVO": "Motivo",
    "ESTADO": "Estado",
    "ESTADO_MERCANCIA": "Estado mercanc\u00eda",
    "ESTADO_CAJA": "Estado caja",
    "ESTADO_OLORES": "Estado olores",
    "ESTADO_LIMPIEZA_VEHICULO": "Estado limpieza veh\u00edculo",
    "ESTADO_VEHICULO_SECO": "Estado veh\u00edculo seco",
    "ESTADO_LIBRE_PLAGAS": "Estado libre de plagas",
}

# Ordem exata das colunas no ficheiro (usada para validar o cabecalho
# antes de ler os dados por posicao)
ORDEM_CABECALHO_CARGAS = list(COLUNAS_CARGAS.values())

COLUNA_ORIGEM = "ficheiro_origem"
ANOS = ("2025", "2026")

# Chave unica: TODAS as colunas. Uma linha so e considerada duplicada se
# for igual em tudo - assim nunca se perdem linhas legitimas que
# partilhem so alguns campos (ex: "Ruta" sozinho nao e unico - ha varias
# linhas com o mesmo numero de rota mas dados diferentes).
CHAVE_UNICA_CARGAS = tuple(COLUNAS_CARGAS.keys())

## Passo 1 — Preparar a base de dados

In [9]:
def preparar_base_dados(con):
    """Cria a base de dados (se nao existir), as tabelas cargas_<ano> e
    o indice unico que impede duplicados. O indice e sempre recriado
    (DROP + CREATE) para garantir que reflete a CHAVE_UNICA_CARGAS
    atual, mesmo que ja existisse com outra definicao de uma versao
    anterior deste notebook. Usa COALESCE(..., \'\') porque o SQLite
    trata cada NULL como diferente de outro NULL."""
    colunas_sql = ",\n        ".join(f'"{c}" TEXT' for c in COLUNAS_CARGAS)
    chave_sql = ", ".join(f'COALESCE("{c}", \'\')' for c in CHAVE_UNICA_CARGAS)

    cur = con.cursor()
    for ano in ANOS:
        cur.execute(f'''
            CREATE TABLE IF NOT EXISTS cargas_{ano} (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                {colunas_sql},
                "{COLUNA_ORIGEM}" TEXT
            )
        ''')
        cur.execute(f'DROP INDEX IF EXISTS idx_cargas_{ano}_chave')
        cur.execute(
            f'CREATE UNIQUE INDEX idx_cargas_{ano}_chave '
            f'ON cargas_{ano}({chave_sql})'
        )
    con.commit()


pasta_db = os.path.dirname(DB_PATH)
if pasta_db and not os.path.exists(pasta_db):
    os.makedirs(pasta_db, exist_ok=True)

con = sqlite3.connect(DB_PATH)
preparar_base_dados(con)
con.close()
print("Base de dados, tabela cargas e indice unico prontos.")

Base de dados, tabela cargas e indice unico prontos.


## Passo 2 — Ler os ficheiros da pasta

O ficheiro CRONO e texto separado por tabs (TSV), em ISO-8859-1. O
cabecalho tem 2 colunas chamadas `Temperatura`, por isso nao e seguro
construir um dict nome->valor - a leitura confia na **posicao** das
colunas, validando primeiro que o cabecalho e exatamente o esperado.

In [10]:
def listar_ficheiros_excel(pasta):
    """Procura ficheiros .xls (tambem em subpastas), sem duplicar por
    causa de maiusculas/minusculas no caminho."""
    encontrados = glob.glob(os.path.join(pasta, "**", "*.xls"), recursive=True)
    vistos = set()
    ficheiros = []
    for caminho in encontrados:
        chave = os.path.normcase(os.path.abspath(caminho))
        if chave not in vistos:
            vistos.add(chave)
            ficheiros.append(caminho)
    return sorted(ficheiros)


def normalizar_valores(valores):
    """Remove espacos a mais em todos os valores de texto - sem isto, a
    mesma linha podia ficar com uma chave diferente entre ficheiros
    (ex: \'PT-1\' vs \'PT-1 \') e deixar de ser reconhecida como
    duplicada."""
    return [v.strip() if isinstance(v, str) else v for v in valores]


def extrair_ano_de_data(valor):
    """Devolve o ano (4 digitos) de uma data no formato DD/MM/AAAA, ou
    None se o valor for vazio/invalido."""
    if valor and str(valor).strip().upper() != "N/D":
        partes = str(valor).strip().split("/")
        if len(partes) == 3 and partes[2].isdigit():
            return partes[2]
    return None


def ler_linhas_csv_cargas(caminho_ficheiro):
    """Le um ficheiro CRONO (texto tab-separated, ISO-8859-1) e devolve
    (cabecalho, lista_de_linhas)."""
    with open(caminho_ficheiro, encoding="iso-8859-1", newline="") as f:
        linhas_ficheiro = list(csv.reader(f, delimiter="\t"))

    if not linhas_ficheiro:
        return [], []

    cabecalho = linhas_ficheiro[0]
    linhas = linhas_ficheiro[1:]

    n_colunas = len(cabecalho)
    linhas_normalizadas = []
    for valores in linhas:
        if len(valores) < n_colunas:
            valores = valores + [None] * (n_colunas - len(valores))
        elif len(valores) > n_colunas:
            valores = valores[:n_colunas]
        linhas_normalizadas.append(valores)

    return cabecalho, linhas_normalizadas


ficheiros = listar_ficheiros_excel(PASTA_FICHEIROS)
print(f"Ficheiros de cargas encontrados: {len(ficheiros)}")

Ficheiros de cargas encontrados: 23


## Passo 3 — Inserir os dados de cargas (sem duplicar)

In [11]:
def montar_linha_cargas(valores, nome_ficheiro):
    return normalizar_valores(valores) + [nome_ficheiro]


idx_fecha_real_entrega = list(COLUNAS_CARGAS.keys()).index("FECHA_REAL_ENTREGA")
idx_fecha_prevista_entrega = list(COLUNAS_CARGAS.keys()).index("FECHA_PREVISTA_ENTREGA")
idx_fecha_prevista_posicionamiento = list(COLUNAS_CARGAS.keys()).index("FECHA_PREVISTA_POSICIONAMIENTO")


def extrair_ano_e_data_cargas(valores):
    """Devolve (ano, data) a partir de Fecha Real Entrega; se vazia, usa
    Fecha Prevista Entrega; por fim Fecha Prevista Posicionamiento (esta
    ultima esta sempre preenchida). Devolve (None, None) se nenhuma data
    for valida."""
    for idx in (idx_fecha_real_entrega, idx_fecha_prevista_entrega, idx_fecha_prevista_posicionamiento):
        valor = valores[idx]
        ano = extrair_ano_de_data(valor)
        if ano:
            return ano, str(valor).strip()
    return None, None


cols_sql = ", ".join(f'"{c}"' for c in COLUNAS_CARGAS)
placeholders = ", ".join(["?"] * len(COLUNAS_CARGAS))
sql_insercao = {
    ano: (
        f'INSERT OR IGNORE INTO cargas_{ano} ({cols_sql}, "{COLUNA_ORIGEM}") '
        f'VALUES ({placeholders}, ?)'
    )
    for ano in ANOS
}

contagem_novas = {ano: 0 for ano in ANOS}
total_duplicadas = 0
total_sem_data = 0
total_cabecalho_invalido = 0

con = sqlite3.connect(DB_PATH)
cur = con.cursor()

for i, caminho in enumerate(ficheiros, start=1):
    nome_ficheiro = os.path.basename(caminho)

    try:
        cabecalho, linhas = ler_linhas_csv_cargas(caminho)
    except Exception as e:
        print(f"[{i}/{len(ficheiros)}] [ERRO] Falha a ler \'{nome_ficheiro}\': {e}")
        continue

    if cabecalho != ORDEM_CABECALHO_CARGAS:
        print(f"[{i}/{len(ficheiros)}] [AVISO] cabecalho de \'{nome_ficheiro}\' diferente do esperado - ficheiro ignorado.")
        total_cabecalho_invalido += 1
        continue

    novas_ficheiro = {ano: 0 for ano in ANOS}
    duplicadas_ficheiro = 0
    sem_data_ficheiro = 0

    for valores in linhas:
        if all(v is None or str(v).strip() == "" for v in valores):
            continue

        ano, _ = extrair_ano_e_data_cargas(valores)
        if ano not in ANOS:
            sem_data_ficheiro += 1
            continue

        linha = montar_linha_cargas(valores, nome_ficheiro)
        cur.execute(sql_insercao[ano], linha)
        if cur.rowcount == 1:
            novas_ficheiro[ano] += 1
        else:
            duplicadas_ficheiro += 1

    con.commit()

    agora = datetime.now().isoformat(timespec="seconds")
    print(f"[{i}/{len(ficheiros)}] {nome_ficheiro} -> "
          f"{novas_ficheiro['2025']} novas 2025, {novas_ficheiro['2026']} novas 2026, "
          f"{duplicadas_ficheiro} ja existiam, {sem_data_ficheiro} sem data valida "
          f"| {agora}")

    for ano in ANOS:
        contagem_novas[ano] += novas_ficheiro[ano]
    total_duplicadas += duplicadas_ficheiro
    total_sem_data += sem_data_ficheiro

con.close()

print("=" * 70)
print("RESUMO FINAL - CARGAS (CRONO)")
print("=" * 70)
print(f"Ficheiros processados:                 {len(ficheiros)}")
print(f"Ficheiros com cabecalho invalido:       {total_cabecalho_invalido}")
print(f"Linhas novas inseridas em 2025:         {contagem_novas['2025']}")
print(f"Linhas novas inseridas em 2026:         {contagem_novas['2026']}")
print(f"Linhas ja existentes (nao inseridas):   {total_duplicadas}")
print(f"Linhas sem data valida:                 {total_sem_data}")
print("=" * 70)

[1/23] crono_1784655922836.xls -> 0 novas 2025, 0 novas 2026, 1023 ja existiam, 0 sem data valida | 2026-08-08T23:15:40
[2/23] crono_1784655960761.xls -> 0 novas 2025, 0 novas 2026, 1152 ja existiam, 0 sem data valida | 2026-08-08T23:15:40
[3/23] crono_1784655998553.xls -> 0 novas 2025, 0 novas 2026, 1113 ja existiam, 0 sem data valida | 2026-08-08T23:15:40
[4/23] crono_1784656022355.xls -> 0 novas 2025, 0 novas 2026, 1171 ja existiam, 0 sem data valida | 2026-08-08T23:15:40
[5/23] crono_1784656044889.xls -> 0 novas 2025, 0 novas 2026, 1237 ja existiam, 0 sem data valida | 2026-08-08T23:15:40
[6/23] crono_1784656070233.xls -> 0 novas 2025, 0 novas 2026, 1265 ja existiam, 0 sem data valida | 2026-08-08T23:15:40
[7/23] crono_1784656086169.xls -> 0 novas 2025, 0 novas 2026, 1171 ja existiam, 0 sem data valida | 2026-08-08T23:15:40
[8/23] crono_1784663524006.xls -> 0 novas 2025, 0 novas 2026, 1699 ja existiam, 134 sem data valida | 2026-08-08T23:15:40
[9/23] crono_1784663538828.xls -> 0 no

## Passo 4 — Validação final

Compara, por mês, o que está nos ficheiros com o que está realmente na
base de dados. A coluna **"Em falta"** deve ficar a 0 em todos os meses -
se não ficar, aponta exatamente onde procurar (ficheiro em falta na
pasta, ou importação ainda não corrida sobre ele).

In [12]:
def obter_chaves_bd(con, nome_tabela, chave):
    colunas_sql = ", ".join(f'COALESCE("{c}", \'\')' for c in chave)
    cur = con.cursor()
    cur.execute(f'SELECT {colunas_sql} FROM {nome_tabela}')
    return set(cur.fetchall())


def validar_cargas(ficheiros, con):
    colunas_ordem = list(COLUNAS_CARGAS.keys())
    chaves_bd = {ano: obter_chaves_bd(con, f"cargas_{ano}", CHAVE_UNICA_CARGAS) for ano in ANOS}
    chaves_por_mes = {}

    for caminho in ficheiros:
        cabecalho, linhas = ler_linhas_csv_cargas(caminho)
        if cabecalho != ORDEM_CABECALHO_CARGAS:
            continue

        for valores in linhas:
            if all(v is None or str(v).strip() == "" for v in valores):
                continue

            ano, data = extrair_ano_e_data_cargas(valores)
            if ano not in ANOS:
                continue

            mes_ano = "/".join(data.split("/")[1:3])

            linha = montar_linha_cargas(valores, os.path.basename(caminho))
            chave = tuple(str(linha[colunas_ordem.index(c)] or "") for c in CHAVE_UNICA_CARGAS)

            entrada = chaves_por_mes.setdefault(mes_ano, {"ano": ano, "chaves": set()})
            entrada["chaves"].add(chave)

    colunas = ["Mes/Ano", "Nos ficheiros", "Na BD", "Em falta"]
    if not chaves_por_mes:
        return pd.DataFrame(columns=colunas)

    resultado = []
    for mes_ano, dados in chaves_por_mes.items():
        chaves_bd_ano = chaves_bd[dados["ano"]]
        total = len(dados["chaves"])
        na_bd = len(dados["chaves"] & chaves_bd_ano)
        resultado.append({"Mes/Ano": mes_ano, "Nos ficheiros": total, "Na BD": na_bd, "Em falta": total - na_bd})

    df = pd.DataFrame(resultado)
    df["_ord"] = df["Mes/Ano"].apply(lambda m: m.split("/")[::-1])
    return df.sort_values("_ord").drop(columns="_ord").reset_index(drop=True)


con = sqlite3.connect(DB_PATH)
resultado = validar_cargas(ficheiros, con)
con.close()

print(f"Cargas: {resultado['Em falta'].sum() if len(resultado) else 0} linhas em falta no total.")
resultado

Cargas: 0 linhas em falta no total.


,Mes/Ano,Nos ficheiros,Na BD,Em falta
0,01/2025,2136,2136,0
1,02/2025,2032,2032,0
2,03/2025,2141,2141,0
3,04/2025,1271,1271,0
4,05/2025,2,2,0
5,12/2025,325,325,0
6,01/2026,2846,2846,0
7,02/2026,2930,2930,0
8,03/2026,3333,3333,0
9,04/2026,3300,3300,0
